In [4]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
import os

# Configure the plotting routines
import pandas as pd

# Import CRUD Python module
from animal_shelter import AnimalShelter

# Configure JupyterDash
JupyterDash.infer_jupyter_proxy_config()

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "SNHU"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Retrieve all documents
df = pd.DataFrame.from_records(db.read({}))

# Drop MongoDB ObjectId column so Dash table does not crash
if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)

# Make sure location fields are numeric
if "location_lat" in df.columns:
    df["location_lat"] = pd.to_numeric(df["location_lat"], errors="coerce")

if "location_long" in df.columns:
    df["location_long"] = pd.to_numeric(df["location_long"], errors="coerce")


###########################
# Rescue Filter Queries
###########################

def get_filter_query(filter_type):
    """Return MongoDB query based on selected rescue type."""

    if filter_type == "Water Rescue":
        return {
            "animal_type": "Dog",
            "breed": {
                "$regex": "Labrador Retriever|Chesapeake Bay Retriever|Newfoundland",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == "Mountain or Wilderness Rescue":
        return {
            "animal_type": "Dog",
            "breed": {
                "$regex": "German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == "Disaster or Individual Tracking":
        return {
            "animal_type": "Dog",
            "breed": {
                "$regex": "Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    # Reset / All Animals
    return {}


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Add Grazioso Salvare logo
image_filename = "Grazioso Salvare Logo.png"

encoded_image = None
if os.path.exists(image_filename):
    encoded_image = base64.b64encode(open(image_filename, "rb").read()).decode()

header_children = []

if encoded_image:
    header_children.append(
        html.Img(
            src="data:image/png;base64,{}".format(encoded_image),
            style={
                "height": "120px",
                "display": "block",
                "margin-left": "auto",
                "margin-right": "auto"
            }
        )
    )

header_children.extend([
    html.Center(html.B(html.H1("Grazioso Salvare Rescue Animal Dashboard"))),
    html.Center(html.H3("Created by Maximo Winfield")),
])

app.layout = html.Div([
    html.Div(header_children),

    html.Hr(),

    html.Div([
        html.Label(
            "Select Rescue Type:",
            style={
                "fontWeight": "bold",
                "fontSize": "18px"
            }
        ),

        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Reset / All Animals", "value": "Reset"},
                {"label": "Water Rescue", "value": "Water Rescue"},
                {"label": "Mountain or Wilderness Rescue", "value": "Mountain or Wilderness Rescue"},
                {"label": "Disaster or Individual Tracking", "value": "Disaster or Individual Tracking"}
            ],
            value="Reset",
            labelStyle={
                "display": "inline-block",
                "margin-right": "20px",
                "margin-top": "10px"
            }
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id="datatable-id",
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict("records"),

        # User-friendly table features
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        column_selectable="single",
        selected_rows=[0],
        selected_columns=[],

        style_table={
            "overflowX": "auto"
        },
        style_cell={
            "textAlign": "left",
            "minWidth": "100px",
            "maxWidth": "220px",
            "whiteSpace": "normal"
        },
        style_header={
            "fontWeight": "bold",
            "backgroundColor": "#D2F3FF"
        }
    ),

    html.Br(),
    html.Hr(),

    # Dashboard charts side-by-side
    html.Div(
        className="row",
        style={"display": "flex"},
        children=[
            html.Div(
                id="graph-id",
                className="col s12 m6",
                style={
                    "width": "50%",
                    "padding": "10px"
                }
            ),
            html.Div(
                id="map-id",
                className="col s12 m6",
                style={
                    "width": "50%",
                    "padding": "10px"
                }
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):
    """Update the data table based on the selected rescue filter."""

    query = get_filter_query(filter_type)
    filtered_df = pd.DataFrame.from_records(db.read(query))

    if "_id" in filtered_df.columns:
        filtered_df.drop(columns=["_id"], inplace=True)

    return filtered_df.to_dict("records")


@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):
    """Display breed distribution based on the filtered data table."""

    if viewData is None or len(viewData) == 0:
        return [
            html.H3("Breed Distribution"),
            html.P("No data available for the selected filter.")
        ]

    dff = pd.DataFrame.from_dict(viewData)

    if "breed" not in dff.columns or dff.empty:
        return [
            html.H3("Breed Distribution"),
            html.P("Breed data is not available.")
        ]

    breed_counts = dff["breed"].value_counts().head(10).reset_index()
    breed_counts.columns = ["breed", "count"]

    return [
        dcc.Graph(
            figure=px.pie(
                breed_counts,
                names="breed",
                values="count",
                title="Top 10 Breeds in Current Selection"
            )
        )
    ]


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):
    """Highlight selected columns in the data table."""

    if selected_columns is None:
        return []

    return [{
        "if": {"column_id": i},
        "background_color": "#D2F3FF"
    } for i in selected_columns]


@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):
    """Update the geolocation map based on the selected animal row."""

    if viewData is None or len(viewData) == 0:
        return [
            html.H3("Animal Location"),
            html.P("No location data available.")
        ]

    dff = pd.DataFrame.from_dict(viewData)

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    if row >= len(dff):
        row = 0

    if "location_lat" not in dff.columns or "location_long" not in dff.columns:
        return [
            html.H3("Animal Location"),
            html.P("Location columns are not available.")
        ]

    lat = dff.iloc[row]["location_lat"]
    lon = dff.iloc[row]["location_long"]

    if pd.isna(lat) or pd.isna(lon):
        return [
            html.H3("Animal Location"),
            html.P("Selected animal does not have valid location coordinates.")
        ]

    animal_name = dff.iloc[row].get("name", "Unknown")
    animal_breed = dff.iloc[row].get("breed", "Unknown breed")

    return [
        dl.Map(
            style={
                "width": "100%",
                "height": "500px"
            },
            center=[lat, lon],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(animal_breed),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(animal_name),
                            html.H2("Breed"),
                            html.P(animal_breed)
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result in JupyterLab mode
app.run_server()

Dash app running on https://streetdata-miamihazard-3000.codio.io/proxy/8050/
